In [1]:
import numpy as np


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torchvision import transforms,models
from torch.utils.data import DataLoader, Dataset
from torch.utils.data import random_split
import os
# import libraries
from models.emotion_hyp import pyramid_trans_expr

# import libraries

import numpy as np
import pandas as pd
import torchvision
from torchvision.datasets import ImageFolder
from torchvision.transforms import transforms

import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Subset


from sklearn.model_selection import StratifiedShuffleSplit

from tqdm import tqdm, trange
import matplotlib.pyplot as plt
from facenet_pytorch import MTCNN, InceptionResnetV1, fixed_image_standardization, training
from os.path import basename
#import cv2
#from mtcnn.mtcnn import MTCNN
import torchvision
from torchvision.datasets import ImageFolder
from torchvision.transforms import transforms

import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Subset


from sklearn.model_selection import StratifiedShuffleSplit

from tqdm import tqdm, trange


from os.path import basename


/path/to/miniconda3/envs/Alitrp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from models.ir50 import Backbone as resnet50

class FER_IR50(nn.Module):
    def __init__(self, num_classes=7, drop_ratio=0.4, mode='ir', freeze_backbone=False):
        super().__init__()
        # Build backbone exactly as your code does
        self.backbone = resnet50(num_layers=50, drop_ratio=drop_ratio, mode=mode)

        # Classification head on top of backbone features
        # Your backbone forward returns (B, 49, 1024) = 7x7 spatial tokens of 1024-d
        # We'll global-average pool to (B, 1024) and map to 7 classes.
        self.head = nn.Linear(1024, num_classes)

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def forward(self, x):
        # Backbone returns (B, 49, 1024)
        x = self.backbone(x)                    # (B, 49, 1024)
        B, N, C = x.shape                       # N should be 49, C=1024
        x1 = x.view(B, 7, 7, C).permute(0, 3, 1, 2)  # (B, 1024, 7, 7)
        x = F.adaptive_avg_pool2d(x1, 1).flatten(1)  # (B, 1024)
        logits = self.head(x)                        # (B, 7)
        return logits , x1

In [3]:

trans_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

give the testset directory at below

Class order is sorted alphabetically

In [ ]:
testset =torchvision.datasets.ImageFolder(root = '/path/to/testset',transform = trans_test)
print(len(testset))


In [18]:
test_loader= DataLoader(testset, 4,shuffle=True, num_workers=0, pin_memory=True)

In [ ]:
use_cuda = torch.cuda.is_available()
print(use_cuda)

device = torch.device("cuda:0")
print(device)

cuda:0


In [ ]:
model = FER_IR50(num_classes=7, drop_ratio=0.4, mode='ir', freeze_backbone=False)
model.cuda()

In [ ]:
checkpoint = torch.load('/path/to/model/weights.pth')
#replace 'module' in the weight key with 'backbone'
new_state_dict = {}
for k, v in checkpoint['model_state_dict'].items():
    new_key = k.replace('module.', '')
    new_state_dict[new_key] = v
model.load_state_dict(new_state_dict)

In [9]:
from sklearn.metrics import f1_score
import torch

def check_accuracy(loader, model):
    model.eval()
    num_correct = 0
    num_samples = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            # If you need gradients for a hook or similar, you can re-enable them:
            with torch.set_grad_enabled(True):
                scores, _ = model(x)

            _, predictions = scores.max(1)

            num_correct += (predictions == y).sum().item()
            num_samples += predictions.size(0)

            all_preds.extend(predictions.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    accuracy = num_correct / num_samples * 100
    f1 = f1_score(all_labels, all_preds, average='macro')

    print(f'Got {num_correct} / {num_samples}  Accuracy: {accuracy:.2f}%  F1 (macro): {f1:.4f}')


# Accuracy and F1 score

In [ ]:
check_accuracy(test_loader,model)

Got 2719 / 3068  Accuracy: 88.62%  F1 (macro): 0.8120


# Class-wise performance analysis

In [ ]:
import os
from pathlib import Path
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
test_dl = deviceDataLoader(test_loader, device)
# -------------------------
# Config
# -------------------------
CLASSES = ["Anger", "Disgust", "Fear", "Happiness", "Neutral", "Sadness", "Surprise"]
NUM_CLASSES = len(CLASSES)

WEIGHT_PATHS = [
    r"/path/to/list/of/weights.pth",
  ]

# Make sure these exist in your environment:
# - model (already created)
# - device (e.g., device = torch.device("cuda" if torch.cuda.is_available() else "cpu"))
# - loader (your eval/test dataloader)

# -------------------------
# Loading helper
# -------------------------
def load_weights_into_model(model, weight_path, device):
    ckpt = torch.load(weight_path, map_location=device)

    # Try common checkpoint formats
    if isinstance(ckpt, dict):
        if "model_state_dict" in ckpt:
            state = ckpt["model_state_dict"]
        elif "state_dict" in ckpt:
            state = ckpt["state_dict"]
        elif "model" in ckpt and isinstance(ckpt["model"], dict):
            state = ckpt["model"]
        else:
            # sometimes the dict IS the state dict
            state = ckpt
    else:
        raise ValueError(f"Unexpected checkpoint type: {type(ckpt)}")

    new_state = {}
    for k, v in state.items():
        # remove common wrappers
        k2 = k.replace("module.", "")
        new_state[k2] = v

    missing, unexpected = model.load_state_dict(new_state, strict=False)
    if missing or unexpected:
        print(f"[WARN] {Path(weight_path).name}: missing={len(missing)} unexpected={len(unexpected)}")
    return missing, unexpected

def get_run_name(weight_path: str) -> str:
    # .../checkpoints_of_ir50/<RUN_NAME>/epochXX_....pth
    return Path(weight_path).parent.name

# -------------------------
# Eval helper (per-class accuracy + overall + macro F1)
# -------------------------
@torch.no_grad()
def eval_per_class_accuracy(loader, model, device, num_classes=7):
    model.eval()

    correct_per_class = torch.zeros(num_classes, dtype=torch.long)
    total_per_class   = torch.zeros(num_classes, dtype=torch.long)

    all_preds = []
    all_labels = []

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        scores, _ = model(x)
        preds = scores.argmax(dim=1)

        all_preds.append(preds.detach().cpu())
        all_labels.append(y.detach().cpu())

        # totals per class
        total_per_class += torch.bincount(y, minlength=num_classes).cpu()

        # correct per class
        correct_mask = (preds == y)
        correct_per_class += torch.bincount(y[correct_mask], minlength=num_classes).cpu()

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    per_class_acc = (correct_per_class.float() / total_per_class.clamp_min(1).float()).numpy() * 100.0
    overall_acc = (correct_per_class.sum().item() / total_per_class.sum().clamp_min(1).item()) * 100.0
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    return per_class_acc, overall_acc, macro_f1

# -------------------------
# Run all weights
# -------------------------
results = []
model = model.to(device)

for w in WEIGHT_PATHS:
    run_name = get_run_name(w)

    load_weights_into_model(model, w, device)
    per_class_acc, overall_acc, macro_f1 = eval_per_class_accuracy(test_dl, model, device, NUM_CLASSES)

    row = {"name": run_name, "overall_acc": overall_acc, "macro_f1": macro_f1}
    for cls, acc in zip(CLASSES, per_class_acc):
        row[cls] = acc
    results.append(row)

df = pd.DataFrame(results).set_index("name")

# Put columns in the exact order you want
df = df[CLASSES + ["overall_acc", "macro_f1"]]

# -------------------------
# Nice pandas "table" view (heatmap-like)
# -------------------------
styled = (
    df.style
      .format("{:.2f}")
      .background_gradient(cmap="viridis", subset=CLASSES + ["overall_acc"])
      .background_gradient(cmap="magma", subset=["macro_f1"])
)

display(styled)

# -------------------------
# Pandas plots
# -------------------------

# 1) Heatmap-like view as an image via pandas (uses matplotlib under the hood)
ax = df[CLASSES].plot(kind="bar", figsize=(18, 6))
ax.set_ylabel("Per-class Accuracy (%)")
ax.set_xlabel("Model (run name)")
ax.legend(ncol=4, bbox_to_anchor=(1.02, 1), loc="upper left")
ax.set_title("Per-class Accuracy for Each Checkpoint Group")
ax.figure.tight_layout()

# 2) Overall accuracy comparison
ax2 = df["overall_acc"].sort_values(ascending=False).plot(kind="bar", figsize=(12, 4))
ax2.set_ylabel("Overall Accuracy (%)")
ax2.set_title("Overall Accuracy (sorted)")
ax2.figure.tight_layout()


# Taking samples from each class randomly

In [30]:
import os
import random
from pathlib import Path
from PIL import Image

CLASSES = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


def list_images(folder: Path):
    if not folder.exists():
        return []
    files = []
    for p in folder.iterdir():
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            files.append(p)
    return sorted(files)


def save_as_jpg(src_path: Path, dst_path: Path, jpeg_quality: int = 95):
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    with Image.open(src_path) as im:
        # Handle alpha / palette images safely
        if im.mode in ("RGBA", "LA") or (im.mode == "P" and "transparency" in im.info):
            im = im.convert("RGBA")
            bg = Image.new("RGB", im.size, (255, 255, 255))
            bg.paste(im, mask=im.split()[-1])
            out = bg
        else:
            out = im.convert("RGB")

        out.save(dst_path, format="JPEG", quality=jpeg_quality, optimize=True)


def sample_and_export(
    roots,
    out_root="sampled_output",
    n_per_class=3,
    seed=42,
    jpeg_quality=95,
    strict=True,  # if True: error when class folder missing or not enough images
):
    random.seed(seed)
    out_root = Path(out_root)

    for root in map(Path, roots):
        if not root.exists():
            raise FileNotFoundError(f"Root directory not found: {root}")

        # Output folder per given directory (root)
        root_out = out_root / root.name
        root_out.mkdir(parents=True, exist_ok=True)

        for cls in CLASSES:
            cls_dir = root / cls
            imgs = list_images(cls_dir)

            if strict:
                if not cls_dir.exists():
                    raise FileNotFoundError(f"Missing class folder: {cls_dir}")
                if len(imgs) < n_per_class:
                    raise ValueError(
                        f"Not enough images in {cls_dir} (have {len(imgs)}, need {n_per_class})"
                    )
                chosen = random.sample(imgs, n_per_class)
            else:
                if not cls_dir.exists() or len(imgs) == 0:
                    print(f"[WARN] Skipping missing/empty folder: {cls_dir}")
                    continue
                chosen = imgs if len(imgs) <= n_per_class else random.sample(imgs, n_per_class)

            for i, img_path in enumerate(chosen, start=1):
                # Keep original filename stem, but enforce .jpg output
                # Format: "anger-1_originalname.jpg"
                safe_name = img_path.name  # keep original file name (with extension)
                dst_name = f"{cls}-{i}_{safe_name}"
                dst_path = root_out / dst_name
                dst_path = dst_path.with_suffix(".jpg")  # force .jpg

                save_as_jpg(img_path, dst_path, jpeg_quality=jpeg_quality)

        print(f"Done: {root}  ->  {root_out}")


if __name__ == "__main__":
    # Example usage: put your directories here
    roots = [
        #r"/path/to/augment_rafdb/train_fixed_augmented_only",
        r"/path/to/mix_dataset/mixed_syntface_dataset/train",
        # r"/path/to/storage_erdi/syntface_datasets/DCFace_balanced_10K",
        # r"/path/to/storage_erdi/syntface_datasets/DigiFace_balanced_10K",
        # r"/path/to/storage_erdi/syntface_datasets/EmoNet_Face_Big_balanced_10K",
        # r"/path/to/storage_erdi/syntface_datasets/fineface_images",
        # r"/path/to/storage_erdi/syntface_datasets/finefaceV2_images",
        #r"/path/to/storage_erdi/syntface_datasets/GANmut-F",
        #r"/path/to/storage_erdi/syntface_datasets/GANmut-V",
        #r"/path/to/storage_erdi/syntface_datasets/stable_generated",
    ]

    sample_and_export(
        roots=roots,
        out_root="/path/to/FER/POSTER1/samplesofdatasets",
        n_per_class=3,
        seed=1376,          # change for different random picks
        jpeg_quality=95,
        strict=True,       # set False to skip missing/too-small classes
    )


Done: /path/to/mix_dataset/mixed_syntface_dataset/train  ->  /path/to/FER/POSTER1/samplesofdatasets/train
